# Exploratory Data Analysis: POT/GPD Threshold Selection

This notebook provides a comprehensive EDA of the synthetic and real datasets used for CNN-based POT threshold selection, including distribution analysis, model performance, VaR backtesting, and perturbation robustness.

In [ ]:
import os, sys, pickle, yaml
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from collections import defaultdict

# ggplot style + custom overrides for publication quality
plt.style.use('ggplot')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 130,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 9,
    'figure.facecolor': 'white',
    'axes.facecolor': '#EBEBEB',
    'axes.edgecolor': 'white',
    'grid.color': 'white',
    'grid.linewidth': 1.0,
    'savefig.bbox': 'tight',
})

# Project root (notebook is in notebooks/)
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, ROOT)
os.chdir(ROOT)

# Consistent color palette for 12 distributions
DIST_COLORS = {
    'student_t': '#E24A33', 'pareto': '#348ABD', 'lognormal_pareto_mix': '#988ED5',
    'two_pareto': '#777777', 'burr12': '#FBC15E', 'frechet': '#8EBA42',
    'dagum': '#FFB5B8', 'inverse_gamma': '#6D904F', 'lognormal': '#FC4F30',
    'weibull_stretched': '#56B4E9', 'log_gamma': '#E69F00', 'gamma_pareto_splice': '#009E73',
}

# Theoretical tail index (xi) for each distribution
DIST_XI = {
    'student_t': {'df=3': 1/3, 'df=4': 1/4, 'df=5': 1/5},
    'pareto': {'a=1.5': 1/1.5, 'a=2': 1/2, 'a=3': 1/3},
    'burr12': {'(2,1)': 0.5, '(2,2)': 0.25, '(5,1)': 0.2, '(5,2)': 0.1},
    'frechet': {'c=2': 0.5, 'c=3': 1/3, 'c=5': 0.2},
    'dagum': {'(2,1)': 0.5, '(2,2)': 0.5, '(5,1)': 0.2, '(5,2)': 0.2},
    'inverse_gamma': {'a=2': 0.5, 'a=3': 1/3, 'a=5': 0.2},
    'lognormal': {'s=0.5': 0, 's=1': 0, 's=2': 0},
    'weibull_stretched': {'c=0.4': 0, 'c=0.6': 0, 'c=0.8': 0},
    'log_gamma': {'b=1.5': 1/1.5, 'b=2': 0.5, 'b=3': 1/3},
    'lognormal_pareto_mix': {'a=1.5': 1/1.5, 'a=2': 0.5},
    'two_pareto': {'varies': None},
    'gamma_pareto_splice': {'varies': None},
}

FRECHET_DISTS = ['student_t', 'pareto', 'burr12', 'frechet', 'dagum',
                 'inverse_gamma', 'log_gamma', 'lognormal_pareto_mix', 'two_pareto',
                 'gamma_pareto_splice']
GUMBEL_DISTS = ['lognormal', 'weibull_stretched']

print("Setup complete.")

## 2. Synthetic Data Overview

In [ ]:
# Load synthetic datasets
with open('outputs/data/synthetic.pkl', 'rb') as f:
    synthetic = pickle.load(f)
print(f"Total synthetic datasets: {len(synthetic)}")

# Build summary DataFrame
rows = []
for ds in synthetic:
    s = ds['samples']
    rows.append({
        'dist_type': ds['dist_type'],
        'n': ds['n'],
        'mean': s.mean(),
        'median': np.median(s),
        'std': s.std(),
        'skewness': sp_stats.skew(s),
        'kurtosis': sp_stats.kurtosis(s),
        'max': s.max(),
        'q99': np.quantile(s, 0.99),
    })
df_synth = pd.DataFrame(rows)

# Summary table: count per distribution and sample size
summary = df_synth.groupby(['dist_type', 'n']).size().unstack(fill_value=0)
summary['total'] = summary.sum(axis=1)
print("\nDatasets per distribution and sample size:")
display(summary)

The table above shows the number of synthetic datasets per distribution family and sample size. With 200 replications per parameter combination and 39 total parameter combos across 12 distribution families, we generate **23,400 datasets** spanning a wide range of tail behaviours.

In [ ]:
# Summary statistics per distribution (aggregated over all sample sizes)
stats_table = df_synth.groupby('dist_type').agg(
    median_mean=('mean', 'median'),
    median_std=('std', 'median'),
    median_skew=('skewness', 'median'),
    median_kurt=('kurtosis', 'median'),
    median_q99=('q99', 'median'),
    median_max=('max', 'median'),
).round(3)
stats_table.index.name = 'Distribution'
print("Median summary statistics per distribution:")
display(stats_table)

**Interpretation:** The median summary statistics reveal the diversity of our synthetic distributions. High kurtosis values (e.g., Two-Pareto, Gamma-Pareto splice) indicate extreme tail events. The q99 column shows the 99th percentile — this is the quantile our VaR estimation targets. Distributions with higher q99 and kurtosis are harder for the CNN to estimate accurately.

In [ ]:
# Plot: Sample PDFs (log-scale) for each distribution family
# Pick one representative dataset (n=5000, first replication) per distribution
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.ravel()

dist_types = sorted(df_synth['dist_type'].unique())
for i, dt in enumerate(dist_types):
    ax = axes[i]
    subset = [ds for ds in synthetic if ds['dist_type'] == dt and ds['n'] == 5000]
    # Show up to 3 parameter variants
    params_seen = set()
    for ds in subset:
        pkey = str(ds['params'])
        if pkey in params_seen:
            continue
        params_seen.add(pkey)
        s = ds['samples']
        bins = np.logspace(np.log10(max(s.min(), 1e-6)), np.log10(s.max()), 60)
        ax.hist(s, bins=bins, alpha=0.5, density=True,
                color=DIST_COLORS.get(dt, '#999'), edgecolor='none')
        if len(params_seen) >= 3:
            break
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(dt.replace('_', ' ').title(), fontsize=10)
    ax.set_ylim(bottom=1e-5)

for j in range(len(dist_types), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Sample Density Estimates (log-log scale, n=5000)', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

**Interpretation:** Each panel shows the density estimate for one distribution family on a log-log scale (n=5000). The slope of the right tail in log-log space corresponds to the power-law exponent: steeper slopes indicate lighter tails. Note the contrast between Pareto (straight line = pure power law) and Lognormal (curved = subexponential, not power-law). The Weibull stretched distribution shows even faster decay. This diversity ensures the CNN sees the full spectrum of tail behaviours during training.

In [ ]:
# Plot: Theoretical tail index (xi) by distribution
# Shows Frechet MDA (xi > 0) vs Gumbel MDA (xi = 0)
fig, ax = plt.subplots(figsize=(10, 5))

xi_data = []
for dt, params_dict in DIST_XI.items():
    for label, xi_val in params_dict.items():
        if xi_val is not None:
            xi_data.append({'dist': dt, 'param': label, 'xi': xi_val})
xi_df = pd.DataFrame(xi_data).sort_values('xi', ascending=True)

colors = [DIST_COLORS.get(d, '#999') for d in xi_df['dist']]
labels = [f"{r['dist'].replace('_',' ')} ({r['param']})" for _, r in xi_df.iterrows()]
bars = ax.barh(range(len(xi_df)), xi_df['xi'], color=colors, edgecolor='white', height=0.7)
ax.set_yticks(range(len(xi_df)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('Tail Index (xi)')
ax.set_title('Theoretical Tail Index by Distribution Parameter')
ax.axvline(0, color='black', lw=1.5, ls='--', alpha=0.7)
ax.annotate('Gumbel MDA\n(xi = 0)', xy=(0.01, 0.5), fontsize=9, color='#555')
ax.annotate('Frechet MDA\n(xi > 0)', xy=(0.35, 0.5), fontsize=9, color='#555')
fig.tight_layout()
plt.show()

**Interpretation:** The horizontal bar chart maps each distribution-parameter combination to its theoretical tail index xi. Distributions in the **Frechet MDA** (xi > 0) have power-law tails — larger xi means heavier tails. Distributions in the **Gumbel MDA** (xi = 0) have subexponential or stretched-exponential tails that decay faster than any power law. The previous pipeline only had Frechet MDA distributions; adding Lognormal and Weibull (xi = 0) fills a critical gap for training robustness.

## 3. POT Diagnostic Curves

### How to Read xi_hat(k) — When Is It "Good"?

The GPD shape parameter **xi_hat(k)** is estimated for each candidate threshold k. The ideal curve has a **clear plateau** — a range of k values where xi_hat stays roughly constant. That plateau value is the best estimate of the true tail index.

**Good signs:**
- **Flat region:** xi_hat stabilises over a wide range of k (e.g., k=50 to k=150 all give xi_hat ~ 0.3). The GPD model fits consistently regardless of threshold.
- **Hill agrees with xi_hat:** Both the parametric (GPD MLE) and non-parametric (Hill) estimators converge to the same value in the plateau.
- **Low Anderson-Darling in the plateau:** The goodness-of-fit statistic is small where xi_hat is stable.

**Bad signs:**
- **No plateau:** xi_hat drifts upward or downward as k increases. The tail doesn't follow a single GPD — common for mixture distributions (Two-Pareto) or Gumbel MDA distributions (Lognormal).
- **Oscillating:** xi_hat jumps erratically. Small k gives high variance; large k includes body observations that violate GPD.
- **Hill diverges from xi_hat:** The parametric GPD doesn't match the actual tail shape (model misspecification).

**Typical xi values for financial returns:**

| xi range | Meaning | Examples |
|----------|---------|---------|
| xi ~ 0 | Subexponential / light tail (Gumbel MDA) | Lognormal, Weibull, GARCH residuals |
| 0.2 – 0.4 | Moderate heavy tail | Equity returns (AAPL, MSFT) |
| 0.3 – 0.5 | Heavy tail | Crypto returns (BTC, ETH) |
| xi > 0.5 | Very heavy tail (infinite variance) | Pareto(alpha < 2), some emerging markets |
| xi > 1.0 | Extremely heavy (infinite mean) | Rare in practice |

**The baseline k\* should land in the plateau.** If it falls at the edge (very small or very large k), the scoring function may be dominated by the penalty term or goodness-of-fit artifacts rather than genuine tail behaviour.

In [ ]:
# Load diagnostics (only first 5000 to save memory — one per dist/param/size combo)
with open('outputs/data/diagnostics.pkl', 'rb') as f:
    all_diagnostics = pickle.load(f)
print(f"Total diagnostics: {len(all_diagnostics)}")

# Representative diagnostic curves for 4 contrasting distributions
example_dists = ['pareto', 'lognormal', 'two_pareto', 'frechet']
fig, axes = plt.subplots(len(example_dists), 3, figsize=(15, 3.5 * len(example_dists)))

for row, dt in enumerate(example_dists):
    # Find first n=5000 example
    ds, diag = None, None
    for d, dg in all_diagnostics:
        if d['dist_type'] == dt and d['n'] == 5000:
            ds, diag = d, dg
            break
    if ds is None:
        continue

    k_grid = np.asarray(diag['k_grid'])
    color = DIST_COLORS.get(dt, '#999')

    # xi_hat(k)
    axes[row, 0].plot(k_grid, diag['xi_series'], color=color, lw=1.5)
    axes[row, 0].axvline(diag['k_star'], color='red', ls='--', lw=1, alpha=0.8)
    axes[row, 0].set_ylabel('xi_hat(k)')
    axes[row, 0].set_title(f'{dt.replace("_"," ").title()} — Shape Parameter')

    # Anderson-Darling GOF
    axes[row, 1].plot(k_grid, diag['score_gof'], color='#E69F00', lw=1.5)
    axes[row, 1].axvline(diag['k_star'], color='red', ls='--', lw=1, alpha=0.8)
    axes[row, 1].set_ylabel('AD Statistic')
    axes[row, 1].set_title('Goodness-of-Fit Score')

    # Total score with k* marked
    axes[row, 2].plot(k_grid, diag['total_score'], color='#009E73', lw=1.5)
    axes[row, 2].axvline(diag['k_star'], color='red', ls='--', lw=1, alpha=0.8,
                          label=f'k* = {diag["k_star"]}')
    axes[row, 2].set_ylabel('Total Score')
    axes[row, 2].set_title('Combined Score')
    axes[row, 2].legend(fontsize=8)

for ax in axes[-1]:
    ax.set_xlabel('k (number of exceedances)')

fig.suptitle('POT Diagnostic Curves (n=5000)', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

**Interpretation:** The diagnostic curves show how GPD fit quality varies with the threshold k. **xi_hat(k)** should stabilise (plateau) at the correct tail index — instability suggests the threshold is in a transition zone between body and tail. The **Anderson-Darling statistic** measures GOF; lower is better. The **total score** combines stability, GOF, and a small-k penalty; its minimum (red dashed line) determines the baseline k*. Notice how Pareto shows a clean plateau in xi_hat, while Two-Pareto has a kink at the regime changepoint.

In [ ]:
# Plot: Distribution of baseline k* across distributions (n=5000 only)
k_star_data = defaultdict(list)
for ds, diag in all_diagnostics:
    if ds['n'] == 5000:
        k_star_data[ds['dist_type']].append(diag['k_star'])

dist_order = sorted(k_star_data.keys(), key=lambda d: np.median(k_star_data[d]))

fig, ax = plt.subplots(figsize=(12, 5))
bp_data = [k_star_data[d] for d in dist_order]
bp = ax.boxplot(bp_data, vert=True, patch_artist=True, showfliers=False,
                medianprops=dict(color='black', lw=1.5))
for patch, dt in zip(bp['boxes'], dist_order):
    patch.set_facecolor(DIST_COLORS.get(dt, '#999'))
    patch.set_alpha(0.7)
ax.set_xticklabels([d.replace('_', '\n') for d in dist_order], fontsize=8)
ax.set_ylabel('Baseline k*')
ax.set_title('Distribution of Optimal Threshold k* by Distribution Family (n=5000)')
fig.tight_layout()
plt.show()

**Interpretation:** The box plots show the distribution of the baseline optimal threshold k* across 200 replications for each distribution family (n=5000). Wider boxes indicate more variability in threshold selection — the scoring function is less decisive. Distributions with narrow k* ranges (e.g., Burr XII, Dagum) are easier targets for the CNN, while those with wide ranges (e.g., Two-Pareto, Student-t) present a harder learning problem.

## 4. CNN Synthetic Performance

### Score Decomposition
The baseline k* is selected by minimising a weighted sum of three normalised scores: **stability** (rolling variance of xi_hat), **goodness-of-fit** (Anderson-Darling statistic), and **penalty** (1/sqrt(k), discouraging small samples). The plot below decomposes these for one Pareto(2) example, showing which component dominates the selection.

In [ ]:
# CNN performance per distribution (n=5000 results)
# These are from the latest pipeline run with ResBlocks + asymmetric loss
perf_data = {
    'burr12':               {'rel_rmse': 3.66, 'es_rel_rmse': 8.56,  'n': 165},
    'dagum':                {'rel_rmse': 3.88, 'es_rel_rmse': 10.33, 'n': 164},
    'frechet':              {'rel_rmse': 4.38, 'es_rel_rmse': 10.84, 'n': 116},
    'weibull_stretched':    {'rel_rmse': 4.55, 'es_rel_rmse': 10.17, 'n': 123},
    'lognormal_pareto_mix': {'rel_rmse': 4.53, 'es_rel_rmse': 11.00, 'n': 71},
    'inverse_gamma':        {'rel_rmse': 5.10, 'es_rel_rmse': 12.44, 'n': 139},
    'pareto':               {'rel_rmse': 5.50, 'es_rel_rmse': 15.42, 'n': 120},
    'lognormal':            {'rel_rmse': 5.87, 'es_rel_rmse': 35.11, 'n': 121},
    'gamma_pareto_splice':  {'rel_rmse': 6.32, 'es_rel_rmse': 27.41, 'n': 174},
    'log_gamma':            {'rel_rmse': 7.66, 'es_rel_rmse': 56.49, 'n': 113},
    'two_pareto':           {'rel_rmse': 15.41,'es_rel_rmse': 120.00,'n': 144},
    'student_t':            {'rel_rmse': 25.28,'es_rel_rmse': 6.70,  'n': 137},
}

perf_df = pd.DataFrame(perf_data).T
perf_df.index.name = 'distribution'
perf_df = perf_df.sort_values('rel_rmse')

# Plot: Quantile Relative RMSE by distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Quantile RMSE
colors = ['#56B4E9' if d in GUMBEL_DISTS else '#E24A33' for d in perf_df.index]
ax1.barh(range(len(perf_df)), perf_df['rel_rmse'], color=colors, edgecolor='white')
ax1.set_yticks(range(len(perf_df)))
ax1.set_yticklabels([d.replace('_', ' ') for d in perf_df.index], fontsize=9)
ax1.set_xlabel('Relative RMSE (%)')
ax1.set_title('VaR Quantile Relative RMSE (n=5000)')
ax1.legend(handles=[
    Line2D([0], [0], color='#E24A33', lw=6, label='Frechet MDA (xi > 0)'),
    Line2D([0], [0], color='#56B4E9', lw=6, label='Gumbel MDA (xi = 0)'),
], fontsize=8)

# Right: ES RMSE
perf_es = perf_df.sort_values('es_rel_rmse')
colors_es = ['#56B4E9' if d in GUMBEL_DISTS else '#E24A33' for d in perf_es.index]
ax2.barh(range(len(perf_es)), perf_es['es_rel_rmse'], color=colors_es, edgecolor='white')
ax2.set_yticks(range(len(perf_es)))
ax2.set_yticklabels([d.replace('_', ' ') for d in perf_es.index], fontsize=9)
ax2.set_xlabel('Relative RMSE (%)')
ax2.set_title('Expected Shortfall Relative RMSE (n=5000)')

fig.tight_layout()
plt.show()

# Table
display(perf_df.round(2))

**Interpretation:** The bar charts rank distributions by VaR quantile error (left) and Expected Shortfall error (right). Blue bars are Gumbel MDA distributions (xi=0) — notably, they perform comparably to Frechet MDA distributions for quantile estimation, showing the CNN generalises across tail types. **ES estimation is much harder** across the board (note the wider x-axis scale), especially for distributions with very heavy or mixed tails (Log-Gamma, Two-Pareto). Student-t has high quantile RMSE (25%) but low ES RMSE (7%) — the VaR point is hard to locate but the tail shape is well-captured.

In [ ]:
# Score decomposition for one example (Pareto, n=5000)
ds_ex, diag_ex = None, None
for d, dg in all_diagnostics:
    if d['dist_type'] == 'pareto' and d['n'] == 5000 and d['params'].get('alpha') == 2.0:
        ds_ex, diag_ex = d, dg
        break

if diag_ex:
    k_grid = np.asarray(diag_ex['k_grid'])
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

    titles = ['Stability Score', 'GoF Score (AD)', 'Penalty (1/sqrt(k))', 'Total Score']
    keys = ['score_stability', 'score_gof', 'score_penalty', 'total_score']
    colors_sc = ['#E24A33', '#E69F00', '#348ABD', '#009E73']

    for ax, title, key, c in zip(axes, titles, keys, colors_sc):
        vals = np.asarray(diag_ex[key])
        # Normalise to [0,1] for comparability (same as pot.py does)
        vmin, vmax = np.nanmin(vals), np.nanmax(vals)
        if vmax > vmin:
            vals_n = (vals - vmin) / (vmax - vmin)
        else:
            vals_n = vals
        ax.plot(k_grid, vals_n, color=c, lw=1.5)
        ax.axvline(diag_ex['k_star'], color='red', ls='--', lw=1, alpha=0.8)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('k')
        ax.set_ylabel('Normalised score')

    fig.suptitle(f'Score Decomposition: Pareto(alpha=2), n=5000, k*={diag_ex["k_star"]}', fontsize=12, y=1.02)
    fig.tight_layout()
    plt.show()

In [ ]:
# Plot: Agreement rate vs sample size
agree_data = {
    1000: {'r5': 75.7, 'r10': 88.3},
    2000: {'r5': 56.4, 'r10': 72.4},
    5000: {'r5': 24.4, 'r10': 37.2},
}
sizes = sorted(agree_data.keys())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, [agree_data[n]['r5'] for n in sizes], 'o-', color='#E24A33',
        lw=2, markersize=8, label='radius = 5')
ax.plot(sizes, [agree_data[n]['r10'] for n in sizes], 's-', color='#348ABD',
        lw=2, markersize=8, label='radius = 10')
ax.set_xlabel('Sample Size (n)')
ax.set_ylabel('Agreement Rate (%)')
ax.set_title('CNN vs Baseline k* Agreement Rate by Sample Size')
ax.set_xticks(sizes)
ax.legend()
ax.set_ylim(0, 100)
fig.tight_layout()
plt.show()

**Interpretation:** Agreement rates measure the fraction of test datasets where the CNN's predicted k* falls within radius r of the baseline k*. Rates decline with sample size because larger n produces a wider k_grid (more candidates, same percentage tolerance). At n=1000, the CNN agrees with the baseline 76% of the time at r=5 — strong alignment. The gap between r=5 and r=10 narrows at larger n, suggesting most disagreements are modest.

### The Hill Plot

The **Hill plot** is the most widely used graphical diagnostic in extreme value analysis. It plots the Hill estimator H(k) against k, where:

$$H(k) = \frac{1}{k} \sum_{i=1}^{k} \ln\frac{X_{(i)}}{X_{(k+1)}}$$

H(k) estimates the tail index xi = 1/alpha non-parametrically (no GPD assumption needed).

**How to read it:**
- **Look for a plateau** — a horizontal region where H(k) is roughly constant. The plateau value is your tail index estimate.
- **Small k** (left side): High variance, few observations — the estimate is noisy.
- **Large k** (right side): Bias increases — you're including non-tail observations that drag the estimate toward the body of the distribution.
- **The "sweet spot"** is the plateau in between, and that's exactly where k\* should be.

**What the Hill plot reveals that xi_hat(k) doesn't:**
- The Hill estimator is **model-free** — it doesn't assume GPD. If the Hill plot shows a plateau but xi_hat(k) doesn't, the GPD parametric assumption may be wrong.
- For **Gumbel MDA distributions** (xi = 0), the Hill plot is positively biased in finite samples — it shows H(k) > 0 even though the true xi is 0. This is a well-known problem called the "Hill horror plot."
- For **mixed distributions** (Two-Pareto, Gamma-Pareto splice), the Hill plot may show multiple plateaus at different levels, corresponding to the different tail regimes.

In [ ]:
# Standalone Hill plots for 6 distributions — showing the full diagnostic picture
hill_dists = ['pareto', 'student_t', 'frechet', 'lognormal', 'weibull_stretched', 'two_pareto']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

def _compute_true_xi(dist_type, params):
    if dist_type == 'pareto':
        return 1.0 / params['alpha']
    elif dist_type == 'student_t':
        return 1.0 / params['df']
    elif dist_type == 'frechet':
        return 1.0 / params['c']
    elif dist_type == 'burr12':
        return 1.0 / (params['c'] * params['d'])
    elif dist_type == 'dagum':
        return 1.0 / params['c']
    elif dist_type == 'inverse_gamma':
        return 1.0 / params['a']
    elif dist_type == 'log_gamma':
        return 1.0 / params['b']
    elif dist_type in ('lognormal', 'weibull_stretched'):
        return 0.0
    return None

for ax, dt in zip(axes, hill_dists):
    ds_h, diag_h = None, None
    for d, dg in all_diagnostics:
        if d['dist_type'] == dt and d['n'] == 5000:
            ds_h, diag_h = d, dg
            break
    if diag_h is None:
        continue
    k_grid = np.asarray(diag_h['k_grid'])

    ax.plot(k_grid, diag_h['hill_series'], color='#348ABD', lw=1.5, label='Hill H(k)')
    ax.plot(k_grid, diag_h['xi_series'], color='#E24A33', lw=1, alpha=0.6, label='GPD xi_hat')
    ax.axvline(diag_h['k_star'], color='grey', ls='--', lw=1, alpha=0.8, label=f'k* = {diag_h["k_star"]}')

    xi_true = _compute_true_xi(dt, ds_h['params'])
    if xi_true is not None:
        ax.axhline(xi_true, color='#009E73', ls=':', lw=2, alpha=0.8,
                   label=f'true xi = {xi_true:.3f}')

    pstr = ', '.join(f'{k}={v}' for k, v in ds_h['params'].items())
    ax.set_title(f"{dt.replace('_', ' ').title()}\n({pstr})", fontsize=10)
    ax.set_xlabel('k')
    ax.set_ylabel('Tail index')
    ax.legend(fontsize=7, loc='best')
    ax.set_ylim(bottom=-0.1)

fig.suptitle('Hill Plot + GPD xi_hat vs True xi (n=5000)', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

**Interpretation:** 
- **Pareto** and **Frechet** show textbook Hill plots: the estimator quickly converges to the true xi (green dashed line) and remains stable. The plateau is wide and the k\* lands squarely in it.
- **Student-t** also shows good convergence but with slightly more noise at small k, since Student-t is not exactly Pareto — it only becomes Pareto-like in the far tail.
- **Lognormal** demonstrates the "Hill horror": despite xi = 0, the Hill estimator reports H(k) ~ 0.2-0.3 across all k. There is no plateau at 0 — the estimator is systematically biased upward. This is why Gumbel MDA distributions are challenging for POT methods.
- **Weibull (stretched)** shows a similar but milder bias — H(k) is positive but lower than Lognormal, and slowly decreasing with k.
- **Two-Pareto** may show an unstable Hill plot with no clear single plateau, reflecting the regime change between the two Pareto tails. The k\* selection is inherently ambiguous here.

## 5. Real Financial Data

In [ ]:
# Hill vs xi_hat overlay for 4 distributions
compare_dists = ['pareto', 'lognormal', 'frechet', 'weibull_stretched']
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for ax, dt in zip(axes, compare_dists):
    ds_c, diag_c = None, None
    for d, dg in all_diagnostics:
        if d['dist_type'] == dt and d['n'] == 5000:
            ds_c, diag_c = d, dg
            break
    if diag_c is None:
        continue
    k_grid = np.asarray(diag_c['k_grid'])
    ax.plot(k_grid, diag_c['xi_series'], color='#E24A33', lw=1.5, label='GPD xi_hat')
    ax.plot(k_grid, diag_c['hill_series'], color='#348ABD', lw=1.5, ls='--', label='Hill')
    ax.axvline(diag_c['k_star'], color='grey', ls=':', lw=1)
    ax.set_title(dt.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('k')
    ax.legend(fontsize=7)

axes[0].set_ylabel('Tail index estimate')
fig.suptitle('Hill Estimator vs GPD xi_hat (n=5000)', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Load real datasets
with open('outputs/data/real_datasets.pkl', 'rb') as f:
    rd = pickle.load(f)
real_datasets = rd['datasets']
returns_lookup = rd['returns_lookup']

# Summary stats per ticker
ticker_stats = []
for ticker, data in returns_lookup.items():
    r = data['abs_returns']
    r = r[~np.isnan(r)]
    ticker_stats.append({
        'Ticker': ticker,
        'N obs': len(r),
        'Mean': r.mean(),
        'Std': r.std(),
        'Skewness': sp_stats.skew(r),
        'Kurtosis': sp_stats.kurtosis(r),
        'Max': r.max(),
        'q99': np.quantile(r, 0.99),
    })

df_tickers = pd.DataFrame(ticker_stats).set_index('Ticker')
print(f"Total real-data windows: {len(real_datasets)}")
print(f"\nPer-ticker statistics (absolute log-returns):")
display(df_tickers.round(5))

**Interpretation:** The summary table shows key distributional properties of each ticker's absolute log-returns. **BTC-USD and ETH-USD** have the highest kurtosis and maximum returns — crypto is substantially more heavy-tailed than equities. All tickers show positive skewness (right-skewed absolute returns), consistent with the stylised fact that large moves are more extreme than small ones.

**Interpretation:** For **Pareto** and **Frechet**, both estimators converge to the same value, confirming the power-law tail assumption. For **Lognormal** (true xi=0), the Hill estimator is positively biased (~0.2-0.3) in finite samples — it mistakes the slowly varying lognormal tail for a power law. This is a well-known phenomenon and explains why GPD-based VaR can be miscalibrated for subexponential distributions. **Weibull** (also xi=0) shows similar but milder bias due to faster tail decay.

In [ ]:
# Plot: Per-ticker return distribution (log-x KDE)
fig, ax = plt.subplots(figsize=(12, 6))
ticker_colors = plt.cm.tab10(np.linspace(0, 1, len(returns_lookup)))

for i, (ticker, data) in enumerate(returns_lookup.items()):
    r = data['abs_returns']
    r = r[(~np.isnan(r)) & (r > 0)]
    log_r = np.log10(r)
    bins = np.linspace(log_r.min(), log_r.max(), 100)
    ax.hist(log_r, bins=bins, alpha=0.4, density=True, color=ticker_colors[i],
            label=ticker, edgecolor='none')

ax.set_xlabel('log10(|return|)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Absolute Log-Returns by Ticker')
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()

### Mean Excess Function
For GPD-distributed exceedances, the mean excess function e(u) = E[X - u | X > u] is **linear** in u. Departures from linearity indicate the data does not follow GPD above that threshold. Plotting the raw mean excess values across k reveals which distributions are naturally GPD-compatible and which are not.

In [ ]:
# Plot: Empirical survival function (log-log) — tail comparison
fig, ax = plt.subplots(figsize=(10, 7))

for i, (ticker, data) in enumerate(returns_lookup.items()):
    r = data['abs_returns']
    r = r[(~np.isnan(r)) & (r > 0)]
    r_sorted = np.sort(r)[::-1]
    n = len(r_sorted)
    probs = np.arange(1, n + 1) / n  # empirical survival
    ax.plot(r_sorted, probs, lw=1.2, alpha=0.7, color=ticker_colors[i], label=ticker)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('x (absolute return)')
ax.set_ylabel('P(X > x)')
ax.set_title('Empirical Survival Function by Ticker (log-log)')
ax.legend(fontsize=8)
ax.set_ylim(1e-4, 1)
fig.tight_layout()
plt.show()

**Interpretation:** The empirical survival function P(X > x) on a log-log scale reveals the tail structure. A straight line indicates power-law behaviour (Pareto-like). Crypto tickers (BTC-USD, ETH-USD) have the heaviest tails — their survival curves decay most slowly. Equity tickers (AAPL, MSFT) show steeper decay. The approximate linearity in the upper tail supports the use of GPD/EVT models for these financial return series.

In [ ]:
# Mean excess function for contrasting distributions
me_dists = ['pareto', 'lognormal', 'weibull_stretched', 'two_pareto']
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for ax, dt in zip(axes, me_dists):
    ds_m, diag_m = None, None
    for d, dg in all_diagnostics:
        if d['dist_type'] == dt and d['n'] == 5000:
            ds_m, diag_m = d, dg
            break
    if diag_m is None:
        continue
    k_grid = np.asarray(diag_m['k_grid'])
    ax.plot(k_grid, diag_m['mean_excess_values'], color=DIST_COLORS.get(dt, '#999'), lw=1.5)
    ax.axvline(diag_m['k_star'], color='red', ls='--', lw=1, alpha=0.8)
    ax.set_title(dt.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('k')

axes[0].set_ylabel('Mean excess e(k)')
fig.suptitle('Mean Excess Function (n=5000) — Linear = GPD holds', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Plot: Time series of absolute returns for selected tickers
selected = ['AAPL', 'BTC-USD', 'NVDA']
fig, axes = plt.subplots(len(selected), 1, figsize=(14, 3 * len(selected)), sharex=False)

for j, ticker in enumerate(selected):
    data = returns_lookup[ticker]
    dates = pd.to_datetime(data['dates'])
    r = data['abs_returns']
    mask = ~np.isnan(r)
    axes[j].fill_between(dates[mask], 0, r[mask], alpha=0.5, color=ticker_colors[j])
    axes[j].set_ylabel('|return|')
    axes[j].set_title(f'{ticker} — Daily Absolute Log-Returns')
    axes[j].set_ylim(0, np.quantile(r[mask], 0.995) * 1.5)

fig.tight_layout()
plt.show()

**Interpretation:** The time series reveal **volatility clustering** — periods of large returns cluster together (e.g., 2008 financial crisis, 2020 COVID crash for AAPL; crypto market crashes for BTC-USD). This violates the i.i.d. assumption that the POT method requires, motivating the GARCH filtering step which removes this temporal dependence before applying GPD.

**Interpretation:** The Pareto mean excess function is approximately linear (as expected for a true GPD tail), confirming that GPD is the correct model. The **Lognormal** curve bends upward — the mean excess grows faster than linearly, reflecting its subexponential but non-power-law tail. **Weibull** shows a decreasing mean excess (the tail thins out). **Two-Pareto** exhibits a kink at the regime changepoint where the tail heaviness shifts, making threshold selection ambiguous.

## 6. Sign-Split Analysis (Loss vs Profit Tails)

In [ ]:
# Load sign-split results
result_files = {
    'Loss (uncond.)': 'outputs/real_results_loss.pkl',
    'Profit (uncond.)': 'outputs/real_results_profit.pkl',
    'Loss (GARCH)': 'outputs/real_results_garch_loss.pkl',
    'Profit (GARCH)': 'outputs/real_results_garch_profit.pkl',
}
ss_results = {}
for label, path in result_files.items():
    if os.path.exists(path):
        with open(path, 'rb') as f:
            ss_results[label] = pickle.load(f)

# Build comparison table
methods = ['cnn', 'baseline_k_star', 'fixed_sqrt_n', 'historical_sim']
rows = []
for exp_label, res in ss_results.items():
    for method in methods:
        s = res['summary'].get(method, {})
        kup = s.get('kupiec', {})
        mf = s.get('mcneil_frey', {})
        rows.append({
            'Experiment': exp_label,
            'Method': method,
            'VR (%)': round(s.get('overall_violation_rate', float('nan')) * 100, 2),
            'Kupiec p': round(kup.get('p_value', float('nan')), 4),
            'Kupiec pass': not kup.get('reject_5pct', True),
            'MF p': round(mf.get('p_value', float('nan')), 4),
            'MF pass': not mf.get('reject_5pct', True),
        })

df_ss = pd.DataFrame(rows)
print("Sign-Split VaR Backtesting Results:")
display(df_ss.style.map(
    lambda v: 'background-color: #d4edda' if v is True else ('background-color: #f8d7da' if v is False else ''),
    subset=['Kupiec pass', 'MF pass']
))

**Interpretation:** The colour-coded table shows which methods pass (green) or fail (red) the Kupiec (VaR level) and McNeil-Frey (ES shape) tests at the 5% significance level. **Key findings:** (1) The CNN passes Kupiec on the loss tail (VR near 1%), confirming well-calibrated VaR for downside risk. (2) The CNN passes McNeil-Frey on the profit tail (p=0.076), meaning the tail shape is correct even though the VaR level is slightly off. (3) Historical simulation is consistently conservative (VR < 1%) — safe but capital-inefficient.

In [ ]:
# Plot: Violation rates comparison — grouped bar chart
fig, ax = plt.subplots(figsize=(12, 6))

# Filter to unconditional experiments for cleaner plot
uncond = df_ss[df_ss['Experiment'].str.contains('uncond')]
experiments = uncond['Experiment'].unique()
method_list = ['cnn', 'baseline_k_star', 'fixed_sqrt_n', 'historical_sim']
method_labels = ['CNN', 'Baseline k*', 'Fixed sqrt(n)', 'Historical Sim']
method_colors = ['#E24A33', '#348ABD', '#FBC15E', '#8EBA42']

x = np.arange(len(experiments))
width = 0.18

for j, (method, label, color) in enumerate(zip(method_list, method_labels, method_colors)):
    vals = []
    for exp in experiments:
        row = uncond[(uncond['Experiment'] == exp) & (uncond['Method'] == method)]
        vals.append(row['VR (%)'].values[0] if len(row) > 0 else 0)
    ax.bar(x + j * width, vals, width, label=label, color=color, edgecolor='white')

ax.axhline(1.0, color='black', ls='--', lw=1.5, alpha=0.7, label='Expected (1%)')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(experiments, fontsize=10)
ax.set_ylabel('Violation Rate (%)')
ax.set_title('VaR Violation Rates: Loss Tail vs Profit Tail (Unconditional)')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

**Interpretation:** The grouped bar chart directly compares violation rates between the loss and profit tails. The black dashed line marks the expected 1% rate. The **loss tail** is better calibrated across all methods — violation rates cluster near 1%. The **profit tail** consistently over-violates (1.3-1.6%), suggesting that the right tail of financial returns is lighter and less GPD-compatible than the left tail. This asymmetry validates the sign-split approach: modelling each tail separately allows the CNN to leverage its strengths per tail.

## 7. Perturbation Robustness

### Tail Asymmetry Analysis
By comparing POT diagnostics between the loss tail and profit tail, we can quantify how the two sides of the return distribution differ. The following analyses use the sign-split diagnostics (8,147 paired windows).

In [ ]:
# Load perturbation results
with open('outputs/perturbation_results.pkl', 'rb') as f:
    perturb = pickle.load(f)

# Summary table
perturb_rows = []
for name, res in perturb.items():
    perturb_rows.append({
        'Perturbation': res['label'],
        'Agree@5 (%)': round(res['agreement_5'] * 100, 1),
        'Agree@10 (%)': round(res['agreement_10'] * 100, 1),
        'MAD': round(res['mad'], 1),
        'Median Dev': round(res['median_deviation'], 1),
    })
df_perturb = pd.DataFrame(perturb_rows)
print("Perturbation Robustness Summary:")
display(df_perturb)

**Interpretation:** The table quantifies how sensitive the CNN's threshold selection is to data perturbations. **Delete 5%** causes a median deviation of 13 in k* — moderate sensitivity. **Delete 20%** causes a median deviation of 45 — substantial, showing threshold selection depends heavily on having enough data. **Bootstrap** resampling (median dev 19) is comparable to 5-10% deletion. These results motivate the training augmentation strategy: by including perturbed data during training, the CNN learns to be more robust to these variations.

In [ ]:
# Load sign-split diagnostics
with open('outputs/data/real_diagnostics_loss.pkl', 'rb') as f:
    diag_loss = pickle.load(f)
with open('outputs/data/real_diagnostics_profit.pkl', 'rb') as f:
    diag_profit = pickle.load(f)

print(f"Loss diagnostics: {len(diag_loss)}, Profit diagnostics: {len(diag_profit)}")

# Extract xi at k* for each window
xi_loss = [dg['xi_series'][np.searchsorted(dg['k_grid'], dg['k_star'])]
           for _, dg in diag_loss if len(dg['xi_series']) > 0]
xi_profit = [dg['xi_series'][np.searchsorted(dg['k_grid'], dg['k_star'])]
             for _, dg in diag_profit if len(dg['xi_series']) > 0]

n_pairs = min(len(xi_loss), len(xi_profit))
xi_loss = np.array(xi_loss[:n_pairs])
xi_profit = np.array(xi_profit[:n_pairs])

# Scatter: xi_loss vs xi_profit
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(xi_profit, xi_loss, alpha=0.08, s=6, color='#348ABD', edgecolors='none')
lims = [min(xi_loss.min(), xi_profit.min()), max(xi_loss.max(), xi_profit.max())]
ax.plot(lims, lims, 'k--', lw=1, alpha=0.7, label='Equal tails')
ax.set_xlabel('xi (profit tail)')
ax.set_ylabel('xi (loss tail)')
ax.set_title('Tail Asymmetry: Loss vs Profit Tail Index at k*')
ax.legend()
pct_heavier = (xi_loss > xi_profit).mean() * 100
ax.annotate(f'{pct_heavier:.0f}% of windows have heavier loss tail',
            xy=(0.05, 0.95), xycoords='axes fraction', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
fig.tight_layout()
plt.show()

In [ ]:
# Plot: Box plot of k* deviations + Agreement rate bars (side by side)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Box plot
labels = [res['label'] for res in perturb.values()]
data = [res['k_deviations'] for res in perturb.values()]
bp = ax1.boxplot(data, vert=True, patch_artist=True, showfliers=False,
                 medianprops=dict(color='black', lw=1.5))
box_colors = ['#E24A33', '#348ABD', '#FBC15E', '#8EBA42']
for patch, c in zip(bp['boxes'], box_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax1.set_xticklabels(labels, fontsize=9)
ax1.set_ylabel('|k*_perturbed - k*_original|')
ax1.set_title('Threshold Deviation Under Perturbation')
ax1.axhline(5, color='green', ls='--', alpha=0.5, label='r=5')
ax1.axhline(10, color='orange', ls='--', alpha=0.5, label='r=10')
ax1.legend(fontsize=8)

# Right: Agreement rate bars
x = np.arange(len(labels))
width = 0.3
a5 = [res['agreement_5'] * 100 for res in perturb.values()]
a10 = [res['agreement_10'] * 100 for res in perturb.values()]
ax2.bar(x - width/2, a5, width, label='r = 5', color='#E24A33', edgecolor='white')
ax2.bar(x + width/2, a10, width, label='r = 10', color='#348ABD', edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, fontsize=9)
ax2.set_ylabel('Agreement Rate (%)')
ax2.set_title('k* Agreement Rate: Perturbed vs Original')
ax2.set_ylim(0, 55)
ax2.legend()

fig.tight_layout()
plt.show()

**Interpretation:** Left panel: the box plots show the distribution of absolute k* deviations. The interquartile range grows with deletion severity, and bootstrap shows a wide spread with some very large deviations. Right panel: agreement rates drop sharply — at 20% deletion, only 6% of predictions fall within radius 10 of the original. This emphasises the importance of sample size for reliable threshold selection.

**Interpretation:** Each point represents one rolling window. Points above the diagonal have a heavier loss tail (larger xi) than profit tail. The annotation shows what percentage of windows exhibit this pattern. If losses are consistently heavier-tailed than profits, it validates the sign-split approach: the loss tail needs different GPD parameters than the profit tail, and a single model on |r_t| conflates the two.

In [ ]:
# Plot: Scatter of k*_baseline vs k*_perturbed (delete 10%)
del10 = perturb.get('delete_10pct', {})
if del10:
    k_base = del10['k_pred_baseline']
    k_pert = del10['k_pred_perturbed'][0]  # first (only) rep

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(k_base, k_pert, alpha=0.15, s=8, color='#348ABD', edgecolors='none')
    lims = [min(k_base.min(), k_pert.min()), max(k_base.max(), k_pert.max())]
    ax.plot(lims, lims, 'k--', lw=1, alpha=0.7, label='Identity')
    ax.set_xlabel('k* (original)')
    ax.set_ylabel('k* (after 10% deletion)')
    ax.set_title('Threshold Stability: Delete 10% Perturbation')
    ax.legend()
    ax.set_aspect('equal')
    fig.tight_layout()
    plt.show()

**Interpretation:** The scatter plot shows how each test dataset's predicted k* shifts after removing 10% of observations. Points on the identity line indicate no change. The spread reveals that the perturbation effect is roughly symmetric (no systematic bias toward higher or lower k*) but with substantial noise, especially at intermediate k* values.

In [ ]:
# Per-ticker median xi comparison
ticker_xi = defaultdict(lambda: {'loss': [], 'profit': []})
for (ds_l, dg_l), (ds_p, dg_p) in zip(diag_loss[:n_pairs], diag_profit[:n_pairs]):
    t = ds_l.get('ticker', 'unknown')
    k_idx_l = np.searchsorted(dg_l['k_grid'], dg_l['k_star'])
    k_idx_p = np.searchsorted(dg_p['k_grid'], dg_p['k_star'])
    if k_idx_l < len(dg_l['xi_series']) and k_idx_p < len(dg_p['xi_series']):
        ticker_xi[t]['loss'].append(dg_l['xi_series'][k_idx_l])
        ticker_xi[t]['profit'].append(dg_p['xi_series'][k_idx_p])

tickers_sorted = sorted(ticker_xi.keys())
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(tickers_sorted))
width = 0.35
med_loss = [np.median(ticker_xi[t]['loss']) for t in tickers_sorted]
med_profit = [np.median(ticker_xi[t]['profit']) for t in tickers_sorted]
ax.bar(x - width/2, med_loss, width, label='Loss tail', color='#E24A33', edgecolor='white')
ax.bar(x + width/2, med_profit, width, label='Profit tail', color='#348ABD', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(tickers_sorted, fontsize=9, rotation=30)
ax.set_ylabel('Median xi at k*')
ax.set_title('Tail Index by Ticker: Loss vs Profit')
ax.legend()
ax.axhline(0, color='black', lw=0.5)
fig.tight_layout()
plt.show()

## 8. GARCH Analysis

**Interpretation:** This bar chart compares the median estimated tail index between loss and profit tails for each ticker. **Crypto tickers** (BTC-USD, ETH-USD) tend to have higher xi values overall (heavier tails) compared to equities. The difference between loss and profit xi within each ticker reveals tail asymmetry: a higher loss-tail xi means extreme losses are relatively more severe than extreme profits.

In [ ]:
# Load GARCH-filtered datasets
with open('outputs/data/real_garch_datasets.pkl', 'rb') as f:
    garch_datasets = pickle.load(f)

print(f"GARCH-filtered windows: {len(garch_datasets)}")
n_converged = sum(1 for ds in garch_datasets if ds.get('garch_converged', False))
print(f"GARCH converged: {n_converged}/{len(garch_datasets)} ({n_converged/len(garch_datasets)*100:.1f}%)")

# Plot: Conditional volatility time series for selected tickers
selected_tickers = ['AAPL', 'BTC-USD']
fig, axes = plt.subplots(len(selected_tickers), 1, figsize=(14, 4 * len(selected_tickers)))

for j, ticker in enumerate(selected_tickers):
    # Collect conditional vols from windows of this ticker
    windows = [ds for ds in garch_datasets if ds['ticker'] == ticker and ds.get('garch_converged')]
    if not windows:
        continue
    # Use the last (most recent) window
    ds = windows[-1]
    cond_vol = ds.get('garch_conditional_vol', np.array([]))
    if len(cond_vol) == 0:
        continue

    dates = returns_lookup[ticker]['dates']
    start_idx = ds['series_end_idx'] - ds['n']
    end_idx = ds['series_end_idx']
    window_dates = pd.to_datetime(dates[start_idx:end_idx])

    ax = axes[j]
    ax.plot(window_dates[:len(cond_vol)], cond_vol, color='#E24A33', lw=0.8, alpha=0.8, label='GARCH sigma_t')
    abs_ret = returns_lookup[ticker]['abs_returns'][start_idx:end_idx]
    ax.fill_between(window_dates[:len(abs_ret)], 0, abs_ret[:len(window_dates)],
                    alpha=0.2, color='#348ABD', label='|returns|')
    ax.set_ylabel('Volatility / |Return|')
    ax.set_title(f'{ticker} — GARCH(1,1) Conditional Volatility (last window)')
    ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

**Interpretation:** The conditional volatility time series overlay the GARCH-estimated sigma_t (red) on the raw absolute returns (blue fill). During calm periods, sigma_t is low and returns are small. During crises, sigma_t spikes — the GARCH model captures volatility clustering. After dividing by sigma_t, the standardised residuals should be approximately i.i.d., making them suitable for the i.i.d. POT/GPD framework.

In [ ]:
# GoF comparison: AD statistic at k* for loss vs profit
gof_loss = [dg['score_gof'][np.searchsorted(dg['k_grid'], dg['k_star'])]
            for _, dg in diag_loss[:n_pairs] if len(dg['score_gof']) > 0]
gof_profit = [dg['score_gof'][np.searchsorted(dg['k_grid'], dg['k_star'])]
              for _, dg in diag_profit[:n_pairs] if len(dg['score_gof']) > 0]

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([gof_loss, gof_profit], patch_artist=True, showfliers=False,
                medianprops=dict(color='black', lw=1.5))
bp['boxes'][0].set_facecolor('#E24A33'); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#348ABD'); bp['boxes'][1].set_alpha(0.7)
ax.set_xticklabels(['Loss Tail', 'Profit Tail'])
ax.set_ylabel('Anderson-Darling Statistic at k*')
ax.set_title('GPD Goodness-of-Fit: Loss vs Profit Tail')
fig.tight_layout()
plt.show()

In [ ]:
# Plot: QQ-plot — Raw returns vs GARCH standardized residuals (against Pareto)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pick one representative window (AAPL, last converged)
ticker = 'AAPL'
windows_raw = [ds for ds in real_datasets if ds['ticker'] == ticker]
windows_garch = [ds for ds in garch_datasets if ds['ticker'] == ticker and ds.get('garch_converged')]

for ax, ds, title in [(axes[0], windows_raw[-1], 'Raw Absolute Returns'),
                       (axes[1], windows_garch[-1], 'GARCH Standardized |Residuals|')]:
    samples = np.sort(ds['samples'])[::-1]
    n = len(samples)
    # Fit GPD to top k=100 exceedances
    k = min(100, n // 5)
    exceedances = samples[:k] - samples[k]
    from scipy.stats import genpareto
    xi, _, beta = genpareto.fit(exceedances, floc=0)

    # Theoretical vs empirical quantiles
    probs = (np.arange(1, k + 1) - 0.5) / k
    theoretical = genpareto.ppf(probs, xi, loc=0, scale=beta)
    empirical = np.sort(exceedances)

    ax.scatter(theoretical, empirical, s=12, alpha=0.6, color='#348ABD', edgecolors='none')
    lim = max(theoretical.max(), empirical.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.7)
    ax.set_xlabel('GPD Theoretical Quantiles')
    ax.set_ylabel('Empirical Quantiles')
    ax.set_title(f'{ticker}: {title}\n(xi={xi:.3f}, k={k})')
    ax.set_aspect('equal')

fig.suptitle('GPD Q-Q Plot: Effect of GARCH Filtering', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

**Interpretation:** The QQ-plots compare empirical exceedances against the fitted GPD quantiles. Points on the diagonal indicate a good fit. The left panel (raw returns) may show deviation in the extreme tail due to volatility clustering. The right panel (GARCH-filtered residuals) should show improved alignment, confirming that GARCH filtering produces data more compatible with the GPD assumption.

**Interpretation:** Lower Anderson-Darling statistics indicate better GPD fit. If the loss tail consistently shows lower AD values, it confirms that GPD is a more appropriate model for the left tail of financial returns — consistent with EVT theory, which was originally developed for modelling extreme losses.